# 05. Implementación de la Candidate Tower (Torre de Candidatos)

Este cuaderno implementa la **Fase 3** del plan de desarrollo para el recomendador Two Towers.
El objetivo es construir y validar la **Candidate Tower**, el componente encargado de procesar la información de los productos del catálogo (atributos categóricos y continuos) y proyectarlos en el mismo espacio vectorial denso de dimensión $D = 128$ que la Query Tower.

### Objetivos:
1. Cargar el catálogo de productos y el dataset de entrenamiento.
2. Preprocesar las variables imputando valores faltantes en campos categóricos y numéricos.
3. Extraer vocabularios únicos de productos, marcas y categorías (familia1, familia2).
4. Configurar el pipeline de datos (`tf.data.Dataset`) para candidatos.
5. Implementar la clase `CandidateTower` heredando de `tf.keras.Model`.
6. Configurar la normalización de variables numéricas (`precio` y `peso_unitario`).
7. Verificar el paso forward del modelo y asegurar la dimensionalidad y estabilidad de la salida.


In [ ]:
import polars as pl
import tensorflow as tf
import tensorflow_recommenders as tfrs
import numpy as np
import os

print("TensorFlow versión:", tf.__version__)
print("TensorFlow Recommenders versión:", tfrs.__version__)


## 1. Carga de Datasets
Cargamos el catálogo de productos procesado y el dataset de entrenamiento para realizar la imputación y modelado.

In [ ]:
prd_path = "data_processed/products_catalog.parquet"
train_path = "data_processed/retrieval_train.parquet"

products_df = pl.read_parquet(prd_path)
train_df = pl.read_parquet(train_path)

print(f"Catálogo de Productos: {products_df.height:,} filas")
print(f"Dataset de Entrenamiento (Pares): {train_df.height:,} filas")


## 2. Preprocesamiento e Imputación de Nulos
Para evitar fallas de compilación o valores numéricos erróneos en el modelo, preprocesamos los atributos del catálogo:
- Variables categóricas: `marca`, `familia1` y `familia2`. Imputamos nulos con cadenas descriptivas.
- Variables continuas: `precio` y `peso_unitario`. Imputamos nulos con 0.0 (en caso de existir).

In [ ]:
# Imputar nulos en Polars
products_df = products_df.with_columns([
    pl.col("marca").fill_null("SIN_MARCA"),
    pl.col("familia1").fill_null("SIN_CATEGORIA"),
    pl.col("familia2").fill_null("SIN_SUBCATEGORIA"),
    pl.col("precio").fill_null(0.0),
    pl.col("peso_unitario").fill_null(0.0)
])

# Verificar que no queden nulos en estas columnas críticas
null_counts = products_df.select([
    "id_producto", "marca", "familia1", "familia2", "precio", "peso_unitario"
]).null_count()
print("Nulos por columna después de la imputación:")
print(null_counts)


## 3. Extracción de Vocabularios Únicos
Para las capas categóricas `StringLookup`, extraemos vocabularios únicos usando Polars y los convertimos en listas de Python. Esto optimiza el consumo de memoria y evita llamadas pesadas a `.adapt()` en TensorFlow.

In [ ]:
vocab_products = products_df["id_producto"].unique().to_list()
vocab_marca = products_df["marca"].unique().to_list()
vocab_familia1 = products_df["familia1"].unique().to_list()
vocab_familia2 = products_df["familia2"].unique().to_list()

print("Estadísticas de Vocabularios:")
print(f" - Productos únicos (SKUs): {len(vocab_products):,}")
print(f" - Marcas únicas: {len(vocab_marca):,}")
print(f" - Categorías superiores (familia1): {len(vocab_familia1):,}")
print(f" - Subcategorías (familia2): {len(vocab_familia2):,}")


## 4. Pipeline de Datos en TensorFlow (tf.data.Dataset)
Creamos un pipeline eficiente usando `tf.data.Dataset` para alimentar la torre de candidatos.

In [ ]:
def make_candidate_tf_dataset(df: pl.DataFrame, batch_size: int = 1024) -> tf.data.Dataset:
    inputs = {
        "id_producto": df["id_producto"].to_numpy(),
        "marca": df["marca"].to_numpy(),
        "familia1": df["familia1"].to_numpy(),
        "familia2": df["familia2"].to_numpy(),
        "precio": df["precio"].to_numpy().astype(np.float32),
        "peso_unitario": df["peso_unitario"].to_numpy().astype(np.float32),
    }
    
    ds = tf.data.Dataset.from_tensor_slices(inputs)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# Crear dataset de prueba
candidate_ds = make_candidate_tf_dataset(products_df, batch_size=512)
example_batch = next(iter(candidate_ds))

print("Estructura del lote de ejemplo para candidatos:")
print("Llaves:", list(example_batch.keys()))
print("Dimensiones de id_producto:", example_batch["id_producto"].shape)


## 5. Arquitectura de la Candidate Tower (Keras Model)
Definimos la Candidate Tower heredando de `tf.keras.Model`.
Esta torre:
1. Mapea y proyecta variables categóricas (`id_producto`, `marca`, `familia1`, `familia2`) usando capas `StringLookup` y `Embedding`.
2. Concatena y normaliza variables numéricas (`precio` y `peso_unitario`) con `tf.keras.layers.Normalization`.
3. Concatena todos los embeddings y las variables continuas normalizadas en un vector unificado.
4. Procesa esta representación mediante una MLP profunda (red densa con ReLU y Dropout).
5. Emplea una capa final lineal de proyección para generar el embedding de candidato de dimensión $D = 128$.

In [ ]:
class CandidateTower(tf.keras.Model):
    def __init__(
        self,
        vocab_products,
        vocab_marca,
        vocab_familia1,
        vocab_familia2,
        embedding_dim: int = 128,
        dropout_rate: float = 0.2
    ):
        super().__init__()
        
        # 1. Embeddings para Categóricas
        # SKU (id_producto)
        self.product_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_products, mask_token=None)
        self.product_embedding = tf.keras.layers.Embedding(
            input_dim=len(vocab_products) + 1,
            output_dim=64,
            name="candidate_product_emb"
        )
        
        # Marca
        self.marca_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_marca, mask_token=None)
        self.marca_embedding = tf.keras.layers.Embedding(
            input_dim=len(vocab_marca) + 1,
            output_dim=16,
            name="candidate_marca_emb"
        )
        
        # Familia 1
        self.fam1_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_familia1, mask_token=None)
        self.fam1_embedding = tf.keras.layers.Embedding(
            input_dim=len(vocab_familia1) + 1,
            output_dim=16,
            name="candidate_fam1_emb"
        )
        
        # Familia 2
        self.fam2_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_familia2, mask_token=None)
        self.fam2_embedding = tf.keras.layers.Embedding(
            input_dim=len(vocab_familia2) + 1,
            output_dim=16,
            name="candidate_fam2_emb"
        )
        
        # 2. Normalización de Atributos Continuos
        self.continuous_normalization = tf.keras.layers.Normalization(axis=-1)
        
        # 3. Capas MLP de Interacción Profunda
        self.mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(256, activation="relu"),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(embedding_dim, name="candidate_projection")
        ])
        
    def call(self, inputs):
        # Procesar categóricas
        prod_emb = self.product_embedding(self.product_lookup(inputs["id_producto"]))
        marca_emb = self.marca_embedding(self.marca_lookup(inputs["marca"]))
        fam1_emb = self.fam1_embedding(self.fam1_lookup(inputs["familia1"]))
        fam2_emb = self.fam2_embedding(self.fam2_lookup(inputs["familia2"]))
        
        # Procesar continuas: Concatenar precio y peso y normalizar
        precio = tf.expand_dims(inputs["precio"], axis=-1)
        peso = tf.expand_dims(inputs["peso_unitario"], axis=-1)
        continuous_features = tf.concat([precio, peso], axis=-1)
        continuous_norm = self.continuous_normalization(continuous_features)
        
        # Concatenar todos los embeddings y características continuas normalizadas
        concatenated = tf.concat([
            prod_emb,
            marca_emb,
            fam1_emb,
            fam2_emb,
            continuous_norm
        ], axis=-1)
        
        # Pasar por la MLP
        return self.mlp(concatenated)


## 6. Inicialización del Modelo y Adaptación de Normalización
Instanciamos el modelo de Candidate Tower con dimensión de embedding final $D = 128$ y adaptamos la capa de normalización empleando las medias y varianzas de precio y peso de todo el catálogo de productos.

In [ ]:
# Instanciar la Candidate Tower
candidate_tower = CandidateTower(
    vocab_products=vocab_products,
    vocab_marca=vocab_marca,
    vocab_familia1=vocab_familia1,
    vocab_familia2=vocab_familia2,
    embedding_dim=128,
    dropout_rate=0.2
)

# Adaptar la capa de normalización continua con datos del catálogo
continuous_train = products_df.select(["precio", "peso_unitario"]).to_numpy().astype(np.float32)
candidate_tower.continuous_normalization.adapt(continuous_train)

print("Capa de normalización adaptada con éxito.")
print("Media calculada (Precio, Peso Unitario):", candidate_tower.continuous_normalization.mean.numpy())
print("Varianza calculada (Precio, Peso Unitario):", candidate_tower.continuous_normalization.variance.numpy())


## 7. Validación de Inferencia (Forward Pass)
Ejecutamos el paso forward sobre nuestro lote de ejemplo para asegurar estabilidad de cálculo (sin NaNs), verificar que la salida posea la dimensión `(batch_size, D)` y comprobar la compilación correcta.

In [ ]:
# Ejecutar forward pass con el lote de ejemplo
candidate_embeddings = candidate_tower(example_batch)

print("Verificación de Dimensiones:")
for k, v in example_batch.items():
    print(f" - Input '{k}': {v.shape}")
print(f"\n- Salida de la Candidate Tower: {candidate_embeddings.shape}")

# Comprobar estabilidad numérica
num_nans = tf.reduce_sum(tf.cast(tf.math.is_nan(candidate_embeddings), tf.int32)).numpy()
print(f"- Cantidad de valores NaN en la salida: {num_nans}")

# Validar assertions
assert candidate_embeddings.shape == (512, 128), "Dimensiones incorrectas en la proyección del embedding del candidato."
assert num_nans == 0, "Se han generado valores numéricos inestables (NaN)."

print("\n¡Validación completada con éxito! La Candidate Tower funciona según las especificaciones de diseño.")


## 8. Resumen de Parámetros del Modelo
Visualizamos la estructura de parámetros y capas del modelo para verificar la composición del número de parámetros.

In [ ]:
# Llamada dummy para inicializar pesos y mostrar resumen
_ = candidate_tower(example_batch)
candidate_tower.summary()
